## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

## 2. Load the dataset

In [ ]:
df = pd.read_csv("uber.csv")
df.head()
df.shape

## 3. Inspect the data

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## 4. Preprocessing
### 4.1 Drop ID columns and missing values

In [ ]:
df = df.drop(columns=["Unnamed: 0", "key"], errors="ignore")
df = df.dropna()
df.shape

### 4.2 Extract date and time features

In [ ]:
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df = df.dropna(subset=["pickup_datetime"])

df["hour"] = df["pickup_datetime"].dt.hour
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month
df["year"] = df["pickup_datetime"].dt.year
df["dayofweek"] = df["pickup_datetime"].dt.dayofweek
df = df.drop(columns=["pickup_datetime"])
df.head()

### 4.3 Calculate trip distance (Haversine formula)

In [ ]:
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(df["pickup_longitude"], df["pickup_latitude"],
                              df["dropoff_longitude"], df["dropoff_latitude"])
df[["distance_km", "fare_amount"]].head()

### 4.4 Remove impossible values

In [ ]:
df = df[(df["fare_amount"] > 0) &
        (df["distance_km"] > 0) &
        (df["passenger_count"].between(1, 6)) &
        (df["pickup_latitude"].between(-90, 90)) &
        (df["dropoff_latitude"].between(-90, 90)) &
        (df["pickup_longitude"].between(-180, 180)) &
        (df["dropoff_longitude"].between(-180, 180))]
df.shape

## 5. Outlier detection
### 5.1 Boxplots before removing outliers

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); sns.boxplot(y=df["fare_amount"]); plt.title("Fare (before)")
plt.subplot(1, 2, 2); sns.boxplot(y=df["distance_km"]); plt.title("Distance (before)")
plt.show()

### 5.2 Remove outliers with the IQR method

In [ ]:
for col in ["fare_amount", "distance_km"]:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

print("Rows after cleaning:", len(df))

### 5.3 Boxplots after removing outliers

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); sns.boxplot(y=df["fare_amount"]); plt.title("Fare (after)")
plt.subplot(1, 2, 2); sns.boxplot(y=df["distance_km"]); plt.title("Distance (after)")
plt.show()

## 6. Correlation analysis

In [ ]:
corr = df.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
corr["fare_amount"].sort_values(ascending=False)

## 7. Train/test split

In [ ]:
# Uncomment the next line if Random Forest is too slow on the lab PC
# df = df.sample(50000, random_state=42)

X = df[["distance_km", "passenger_count", "hour", "day", "month", "year", "dayofweek"]]
y = df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 8. Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

## 9. Random Forest Regression

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

## 10. Evaluation: R² and RMSE

In [ ]:
def evaluate(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:20s}  R2 = {r2:.4f}   RMSE = {rmse:.4f}")
    return r2, rmse

r2_lr, rmse_lr = evaluate("Linear Regression", y_test, y_pred_lr)
r2_rf, rmse_rf = evaluate("Random Forest", y_test, y_pred_rf)

## 11. Comparison

In [ ]:
results = pd.DataFrame({"Model": ["Linear Regression", "Random Forest"],
                        "R2": [r2_lr, r2_rf],
                        "RMSE": [rmse_lr, rmse_rf]})
results

## 12. Actual vs predicted

In [ ]:
plot_df = results.set_index("Model")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

plot_df[["R2"]].plot.bar(ax=axes[0], legend=False, color="#5dade2", rot=12)
axes[0].set_title("R2 score (higher is better)")

plot_df[["RMSE"]].plot.bar(ax=axes[1], legend=False, color="#e59866", rot=12)
axes[1].set_title("RMSE (lower is better)")

sample = np.random.RandomState(0).choice(len(y_test), 1500, replace=False)
axes[2].scatter(y_test.iloc[sample], lr.predict(X_test.iloc[sample]),
                s=7, alpha=0.5, label="Linear Regression")
axes[2].scatter(y_test.iloc[sample], rf.predict(X_test.iloc[sample]),
                s=7, alpha=0.5, label="Random Forest")
axes[2].plot([0, 25], [0, 25], "k--", lw=1)
axes[2].set_xlabel("Actual fare ($)")
axes[2].set_ylabel("Predicted fare ($)")
axes[2].set_title("Actual vs predicted")
axes[2].legend()

plt.tight_layout()
plt.show()